# Notebook 1 — Embeddings Playground

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karthikapv-kv/ai-upskill-layer-3/blob/main/notebooks/01_embeddings_playground.ipynb)

**Story so far:** Acme's keyword search can't match *"charged twice"* to *"duplicate debit transactions"*. We need numbers that capture meaning → **embeddings**.

**You'll learn:** generate embeddings · compare them with cosine similarity · read a similarity matrix.

⏱️ **In session (10 min):** E2, E3 · 🏠 **Take-home:** the rest · Runtime: CPU

## 0. Setup

In [ ]:
%pip install -q sentence-transformers==6.1.0

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sentence_transformers import SentenceTransformer, util

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"   # 384 dimensions, ~90 MB, Apache-2.0
model = SentenceTransformer(MODEL_NAME)                 # an HF_TOKEN warning is safe to ignore
print("Ready:", MODEL_NAME)

## 1. Why not one-hot vectors? 🏠
One position per word, a single `1`:

In [ ]:
vocabulary = ["charged", "debited", "refund", "kafka", "rollback"]

def one_hot(word):
    vector = np.zeros(len(vocabulary))
    vector[vocabulary.index(word)] = 1
    return vector

print("charged:", one_hot("charged"))
print("charged · debited =", np.dot(one_hot("charged"), one_hot("debited")))
print("charged · kafka   =", np.dot(one_hot("charged"), one_hot("kafka")))

**Both are 0.** "charged" is as far from "debited" as from "kafka", so there are no relationships. We need learned, dense vectors: embeddings.

## 2. ✏️ E1: Predict first
Things Acme developers say. Which pair is closest? Which is the odd one out?

In [ ]:
billing_sentences = [
    "My customer was charged twice.",
    "A shopper paid two times for the same order.",
    "The customer's card was declined.",
    "Our Kafka consumer stopped processing messages.",
]
how_to_sentences = [
    "How do I reverse a duplicate transaction?",
    "How can I refund a customer who paid twice?",
    "How do I roll back a failed deployment?",
    "How do I change the rate limit for a client?",
]

# ✏️ E1: your guesses (sentence numbers 0–3)
my_prediction_billing = {"closest pair": "?", "odd one out": "?"}
my_prediction_how_to = {"closest pair": "?", "odd one out": "?"}

## 3. Generate embeddings
`embed_texts` gives one row per sentence, each vector scaled to length 1. We'll reuse it in every notebook.

In [ ]:
def embed_texts(texts):
    return model.encode(texts, normalize_embeddings=True)

billing_embeddings = embed_texts(billing_sentences)

### ⏱️ E2: Inspect the embeddings (~3 min)

In [ ]:
# ⏱️ E2
embedding_shape = None      # TODO: shape of billing_embeddings
first_ten_numbers = None    # TODO: first 10 numbers of the first sentence's vector

<details><summary>💡 Hint 1</summary>

`billing_embeddings.shape`

</details>

<details><summary>💡 Hint 2</summary>

`billing_embeddings[0][:10]`

</details>

In [ ]:
if embedding_shape is None or first_ten_numbers is None:
    print("E2 not finished yet: fill in the TODOs above.")
else:
    print("Shape:", embedding_shape)
    print("First 10 numbers:", np.round(first_ten_numbers, 3))
    print("Vector lengths:", np.round(np.linalg.norm(billing_embeddings, axis=1), 3))

**Expected:** `(4, 384)`, small unreadable numbers, lengths `1.0`. No single number means "charged". What matters is how vectors compare.

## 4. Cosine similarity
Do two arrows point the same way? 1 = same, 0 = perpendicular, −1 = opposite.

In [ ]:
examples_2d = {
    "Same direction": ([1, 0], [5, 0]),
    "Similar direction": ([3, 4], [4, 3]),
    "Perpendicular": ([1, 0], [0, 1]),
    "Opposite": ([2, 2], [-1, -1]),
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, (name, (a, b)) in zip(axes, examples_2d.items()):
    ax.quiver(0, 0, *b, angles="xy", scale_units="xy", scale=1, color="tab:orange", label=f"B = {b}")
    ax.quiver(0, 0, *a, angles="xy", scale_units="xy", scale=1, color="tab:blue", label=f"A = {a}",
              width=0.012, zorder=3)
    ax.set(xlim=(-5.5, 5.5), ylim=(-5.5, 5.5), aspect="equal", title=name)
    ax.axhline(0, color="lightgray", lw=0.8)
    ax.axvline(0, color="lightgray", lw=0.8)
    ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\lVert A \rVert \, \lVert B \rVert}$$

In [ ]:
A, B = np.array([3, 4]), np.array([4, 3])
print("dot product:", np.dot(A, B))                           # 3*4 + 4*3 = 24
print("lengths:", np.linalg.norm(A), np.linalg.norm(B))        # 5.0 5.0
print("cosine:", np.dot(A, B) / (np.linalg.norm(A) * np.linalg.norm(B)))   # 0.96

### ⏱️ E3: Implement cosine similarity (~5 min)

In [ ]:
# ⏱️ E3
def cosine_similarity(a, b):
    return None   # TODO: (a · b) / (||a|| * ||b||)

<details><summary>💡 Hint 1</summary>

`np.dot(a, b)` and `np.linalg.norm(a)`

</details>

In [ ]:
if cosine_similarity(np.array([3, 4]), np.array([4, 3])) is None:
    print("E3 not finished yet: fill in the TODO above.")
else:
    for name, (a, b) in examples_2d.items():
        print(f"{name:<18} {cosine_similarity(np.array(a), np.array(b)):6.3f}")
    mine = cosine_similarity(billing_embeddings[0], billing_embeddings[1])
    library = util.cos_sim(billing_embeddings[0], billing_embeddings[1]).item()
    print(f"\n'charged twice' vs 'paid two times' (384 dims): yours {mine:.3f} | library {library:.3f}")

**Expected:** `1.000 · 0.960 · 0.000 · -1.000`, then **≈ 0.558** for both. Same formula in 2D or 384D.

Scores are **relative**: 0.8 ≠ "80% similar", and high ≠ identical.

## 5. Similarity matrix 🏠
Every pair at once. Vectors have length 1, so `embeddings @ embeddings.T` gives all the cosines.

In [ ]:
def plot_similarity(sentences, title):
    vectors = embed_texts(sentences)
    scores = vectors @ vectors.T
    fig, ax = plt.subplots(figsize=(7, 5))
    image = ax.imshow(scores, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(sentences)), [f"[{i}]" for i in range(len(sentences))])
    ax.set_yticks(range(len(sentences)), [f"[{i}] {s}" for i, s in enumerate(sentences)])
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            ax.text(j, i, f"{scores[i, j]:.3f}", ha="center", va="center",
                    color="white" if scores[i, j] > 0.6 else "black")
    ax.set_title(title)
    fig.colorbar(image, ax=ax, fraction=0.046)
    plt.show()

plot_similarity(billing_sentences, "Billing sentences")
plot_similarity(how_to_sentences, "How-to questions")

**Expected (approx.):** charged twice ↔ paid two times **0.558** (same meaning, no shared words) · ↔ card declined **0.386** (same topic) · ↔ Kafka **0.230**.
Reverse duplicate ↔ refund someone who paid twice **0.541** (same intent) · ↔ roll back deployment **0.303** · ↔ rate limit **0.104**.

Similarity mixes **meaning**, **topic** and **intent**. That's how *"charged twice"* will find *"duplicate debit"* in Notebook 2.

## 6. Context matters 🏠
"key" means different things at Acme. The model reads the whole sentence.

In [ ]:
plot_similarity([
    "The idempotency key was reused on every retry.",
    "The client's API key hit the rate limit.",
    "Retries must send the same Idempotency-Key header.",
    "Requests from that client are getting HTTP 429.",
], "Which 'key'?")

**Expected:** idempotency key ↔ Idempotency-Key header **0.717**, but idempotency key ↔ API key only **0.354**. API key ↔ HTTP 429 **0.480**: same meaning of "key" wins.

## 7. Surprise: opposite outcome, high similarity 🏠

In [ ]:
for first, second in [("The deployment succeeded.", "The deployment failed."),
                      ("The payment succeeded.", "The payment failed.")]:
    a, b = embed_texts([first, second])
    print(f"{a @ b:.3f}  {first} ↔ {second}")

**Expected:** ≈ **0.844** and **0.794**. Embeddings capture the *topic* strongly and the exact outcome weakly, so a search for a failed deployment can return success stories.

## 8. *(Optional)* The Acme knowledge base in 2D 🏠
PCA squashes 384D → 2D so we can look at the 18 docs we'll search in Notebook 2. Only a rough picture; trust the numbers, not the plot.

In [ ]:
import json, os, urllib.request
from sklearn.decomposition import PCA

DATA_BASE_URL = "https://raw.githubusercontent.com/karthikapv-kv/ai-upskill-layer-3/main/data"

def data_file(relative_path):
    """Local path to a workshop data file; downloads it from GitHub if needed."""
    for folder in ["data", "../data"]:
        path = os.path.join(folder, relative_path)
        if os.path.exists(path):
            return path
    path = os.path.join("data", relative_path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    urllib.request.urlretrieve(f"{DATA_BASE_URL}/{relative_path}", path)
    return path

with open(data_file("kb_documents.json")) as f:
    documents = json.load(f)

pca = PCA(n_components=2)
points = pca.fit_transform(embed_texts([d["title"] + ". " + d["content"] for d in documents]))
print(f"Variation kept in 2D: {pca.explained_variance_ratio_.sum():.0%}")

fig, ax = plt.subplots(figsize=(9, 6.5))
for category in sorted({d["category"] for d in documents}):
    idx = [i for i, d in enumerate(documents) if d["category"] == category]
    ax.scatter(points[idx, 0], points[idx, 1], s=60, label=category)
for (x, y), d in zip(points, documents):
    ax.annotate(d["document_id"], (x, y), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.legend()
plt.show()

**Expected:** Kafka, API, deployment and payments docs form their own regions. Incidents sit near their topic (KB-017 duplicate charges → payments, KB-018 pool exhaustion → database/API). Only **≈ 27%** of the variation is kept.

## Summary
- One-hot: no relationships. **Embeddings**: learned vectors where related text is close.
- **Cosine similarity** = direction match; read it relatively.
- Similar ≠ same meaning (topic dominates).

➡️ **Notebook 2:** use this to build semantic search over the Acme Pay knowledge base.

---
## Solutions
**E1:** Billing: closest **[0]–[1]** (same meaning, no shared words); odd one out **[3]** (Kafka). How-to: closest **[0]–[1]** (same intent); odd one out **[3]** (rate limit).

In [ ]:
#@title E2 solution
embedding_shape = billing_embeddings.shape
first_ten_numbers = billing_embeddings[0][:10]
print(embedding_shape, np.round(first_ten_numbers, 3))

In [ ]:
#@title E3 solution
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(cosine_similarity(np.array([3, 4]), np.array([4, 3])))   # 0.96